# Lab 3 — Open Microduck in MuJoCo and watch it fall

**EP01 · What is Physical AI?** · Previous video: Lab 2, Python, Git and Colab

In Lab 1 we drove Microduck with a policy in control. In this lab we open **its model ourselves** in MuJoCo,
see which joints, motors and sensors it has, then tell it to simply hold its pose, with no policy.

## Setup

- Install **MuJoCo 3.17.0** (pinned, so results match the video) · NumPy, Pillow and matplotlib are already on Colab.
- Colab has no screen, so we set `MUJOCO_GL=egl` to let MuJoCo render off-screen.
- Microduck's model comes from the `pollen-robotics/microduck_rl` repo at commit `bae39f5` (pinned for the whole term).
- The MJCF files are Apache-2.0 · the 3D files (`*.stl`) are © Pollen Robotics, **CC BY-NC-SA 4.0**.
- How to install and use Colab is in Lab 2.

In [ ]:
%pip install -q mujoco==3.17.0

import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"   # Colab has no screen: render off-screen
import mujoco
print(mujoco.__version__)

!git clone -q https://github.com/pollen-robotics/microduck_rl.git
!git -C microduck_rl checkout -q bae39f5900b318f7e87fc485c30a31213cfa14eb
SCENE = "microduck_rl/src/mjlab_microduck/robot/microduck/scene.xml"

## Open the model: what does Microduck have?

- **model** (`MjModel`) = what doesn't change: shape, masses, joints, motors, sensors
- **data** (`MjData`) = what changes every step: angles, velocities, time, sensor readings

In [ ]:
model = mujoco.MjModel.from_xml_path(SCENE)
data = mujoco.MjData(model)
print("joints:", model.njnt, "| motors:", model.nu, "| sensors:", model.nsensor)

In [ ]:
for i in range(model.njnt):
    print(i, model.joint(i).name)

In [ ]:
for i in range(model.nsensor):
    print(model.sensor(i).name, "->", model.sensor_dim[i], "numbers")

## Experiment: hold the pose, then let go

The model has a saved pose called **STAND** · we put the robot in it, tell every motor to hold its angle, then let physics run for 1.5 seconds.

In [ ]:
import numpy as np
from PIL import Image
renderer = mujoco.Renderer(model, height=360, width=640)
cam = mujoco.MjvCamera()
cam.lookat[:] = [0, 0, 0.08]
cam.distance, cam.azimuth, cam.elevation = 0.6, 135, -15

def snapshot(data):
    renderer.update_scene(data, camera=cam)
    return renderer.render()

mujoco.mj_resetDataKeyframe(model, data, model.key("STAND").id)
mujoco.mj_forward(model, data)
Image.fromarray(snapshot(data))

In [ ]:
times, heights, frames = [], [], []
while data.time < 1.5:
    mujoco.mj_step(model, data)
    times.append(data.time)
    heights.append(data.qpos[2])          # trunk height (m)
    if len(times) % 150 == 0 and len(frames) < 4:
        frames.append(snapshot(data))     # a picture every 0.3 s

heights = np.array(heights)
times = np.array(times)
print("trunk below 9 cm at", times[heights < 0.09][0].round(2), "s")
Image.fromarray(np.hstack(frames))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(times, heights * 100, label="hold STAND (XML servos)")
plt.xlabel("time (s)"); plt.ylabel("trunk height (cm)")
plt.legend(); plt.show()

## Try it: what if the servos were stiffer?

**kp** is servo stiffness: for the same error from the target angle, a higher kp pulls back harder (studied properly in EP05).
The XML sets 0.55 · try 2.0, the value `microduck_rl` uses while the robot stands by (standby).

> ⚠️ This is MuJoCo's simple servo, not the servo model used to train the real policy.

In [ ]:
model.actuator_gainprm[:, 0] = 2.0      # stiffer servos: kp 0.55 -> 2.0
model.actuator_biasprm[:, 1] = -2.0

mujoco.mj_resetDataKeyframe(model, data, model.key("STAND").id)
heights2 = []
while data.time < 1.5:
    mujoco.mj_step(model, data)
    heights2.append(data.qpos[2])
print("lowest trunk height:", round(min(heights2) * 100, 1), "cm")

plt.figure(figsize=(7, 3))
plt.plot(times, heights * 100, label="XML servos (kp 0.55)")
plt.plot(times, np.array(heights2) * 100, label="stiffer servos (kp 2.0)")
plt.xlabel("time (s)"); plt.ylabel("trunk height (cm)")
plt.legend(); plt.show()
Image.fromarray(snapshot(data))

## Summary + assignment

- In the model, Microduck has **14 joints with motors** (5 per leg, 4 in the neck and head) and **IMU** sensors · no sensor directly gives the feet's position
- Holding the standing pose with the XML servos: the trunk drops below 9 cm at **0.86 s** · stiffer servos stand, but still don't balance by themselves

**Class 1 assignment:** write a one-page summary of *what Microduck needs to know and do in order to walk*, using the joint and sensor lists from this lab and what you saw in Lab 1.

**Open question:** how can the robot know where its feet are?

**Next:** EP01 recap